# Vector vs Scalar Diffraction: When Does NA Matter?

This notebook compares the Richards-Wolf vectorial diffraction theory with the scalar Airy pattern approximation at different numerical apertures.

Key question: At what NA do vectorial (polarization) effects become significant?

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import j1
from monte_carlo import RichardsWolfSimulator
from monte_carlo import metrics

sns.set_theme(style="whitegrid", font_scale=1.5)

import os
output_dir = '../data/richards_wolf'
os.makedirs(output_dir, exist_ok=True)

## Define Scalar Airy Pattern

The scalar approximation (valid for low NA) gives the Airy pattern:

$$I(r) = I_0 \left[\frac{2J_1(kr)}{kr}\right]^2$$

where $k = 2\pi \text{NA}/\lambda$

In [ ]:
def scalar_airy_pattern(r, wavelength, NA):
    """
    Compute scalar Airy diffraction pattern.
    
    Parameters
    ----------
    r : np.ndarray
        Radial coordinates in microns
    wavelength : float
        Wavelength in microns
    NA : float
        Numerical aperture
    
    Returns
    -------
    intensity : np.ndarray
        Normalized intensity
    """
    k = 2 * np.pi * NA / wavelength
    kr = k * r
    
    intensity = np.ones_like(kr)
    nonzero = kr != 0
    intensity[nonzero] = (2 * j1(kr[nonzero]) / kr[nonzero])**2
    
    return intensity

## Comparison at Different NAs

Compare vector vs scalar at NA = 0.1, 0.3, 0.5, 0.7, 0.9

In [ ]:
wavelength = 0.532  # microns
NA_list = [0.1, 0.3, 0.5, 0.7, 0.9]

# Storage for results
results = {}

for NA in NA_list:
    print(f"\nComputing NA = {NA}...")
    
    # Create Richards-Wolf simulator
    rw = RichardsWolfSimulator(
        wavelength=wavelength,
        numerical_aperture=NA,
        n_medium=1.0,
        polarization='x'
    )
    
    # Radial coordinates
    r_max = 4 * rw.airy_radius
    r = np.linspace(0, r_max, 100)
    x = r
    y = np.zeros_like(r)
    
    # Vector (Richards-Wolf)
    intensity_vector = rw.focal_plane_intensity_pattern(x, y)
    
    # Scalar (Airy)
    intensity_scalar = scalar_airy_pattern(r, wavelength, NA)
    
    # Store results
    results[NA] = {
        'r': r,
        'vector': intensity_vector,
        'scalar': intensity_scalar,
        'airy_radius': rw.airy_radius
    }
    
    # Compute RMSE
    rmse = metrics.rmse(intensity_vector, intensity_scalar)
    print(f"  RMSE: {rmse:.6f}")
    print(f"  Airy radius: {rw.airy_radius:.4f} μm")

print("\nDone!")

## Plot: Comparison at All NAs

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for i, NA in enumerate(NA_list):
    ax = axes[i]
    
    r = results[NA]['r']
    vector = results[NA]['vector']
    scalar = results[NA]['scalar']
    airy_radius = results[NA]['airy_radius']
    
    # Plot both
    ax.plot(r, scalar, linewidth=4, alpha=0.5, color='tab:orange', 
            label='Scalar (Airy)', zorder=3)
    ax.plot(r, vector, linewidth=3, linestyle='--', color='tab:blue', 
            label='Vector (Richards-Wolf)')
    
    ax.axvline(airy_radius, color='gray', linestyle=':', alpha=0.5, linewidth=1.5)
    
    ax.set_xlabel('Radial distance r (μm)')
    ax.set_ylabel('Normalized Intensity')
    ax.set_title(f'NA = {NA}', fontweight='bold', fontsize=16)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1.1])

# Remove extra subplot
axes[-1].remove()

plt.tight_layout()
plt.savefig(f'{output_dir}/vector_vs_scalar_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Quantitative Comparison: RMSE vs NA

In [ ]:
# Compute metrics for each NA
rmse_list = []
fidelity_list = []

for NA in NA_list:
    vector = results[NA]['vector']
    scalar = results[NA]['scalar']
    
    rmse = metrics.rmse(vector, scalar)
    fid = metrics.fidelity(scalar, vector)
    
    rmse_list.append(rmse)
    fidelity_list.append(fid)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# RMSE
ax = axes[0]
ax.plot(NA_list, rmse_list, 'o-', linewidth=3, markersize=10, color='tab:red')
ax.axhline(0.01, color='gray', linestyle='--', alpha=0.5, label='1% error threshold')
ax.set_xlabel('Numerical Aperture')
ax.set_ylabel('RMSE (Vector vs Scalar)')
ax.set_title('Error vs Numerical Aperture', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Fidelity
ax = axes[1]
ax.plot(NA_list, fidelity_list, 'o-', linewidth=3, markersize=10, color='tab:green')
ax.axhline(0.99, color='gray', linestyle='--', alpha=0.5, label='99% fidelity')
ax.set_xlabel('Numerical Aperture')
ax.set_ylabel('Fidelity (1.0 = perfect match)')
ax.set_title('Fidelity vs Numerical Aperture', fontweight='bold')
ax.set_ylim([0.95, 1.0])
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{output_dir}/metrics_vs_NA.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary Table

In [ ]:
print("=" * 80)
print("VECTOR VS SCALAR COMPARISON")
print("=" * 80)
print(f"{'NA':<10} {'RMSE':<15} {'Fidelity':<15} {'Assessment':<30}")
print("-" * 80)

for NA, rmse, fid in zip(NA_list, rmse_list, fidelity_list):
    if rmse < 0.01:
        assessment = "Scalar approximation valid"
    elif rmse < 0.05:
        assessment = "Minor vectorial effects"
    else:
        assessment = "Significant vectorial effects"
    
    print(f"{NA:<10.1f} {rmse:<15.6f} {fid:<15.6f} {assessment:<30}")

print("=" * 80)
print("\nConclusion:")
print("  - NA < 0.5: Scalar (Airy) approximation is adequate")
print("  - NA > 0.7: Vectorial (Richards-Wolf) theory required")
print("  - NA 0.5-0.7: Transition region")

## Field Component Analysis (High NA)

Show why vectorial effects matter at high NA: the Ez component becomes significant.

In [ ]:
# High NA case
NA_high = 0.9
rw = RichardsWolfSimulator(wavelength=wavelength, numerical_aperture=NA_high, polarization='x')

r = np.linspace(0, 2*rw.airy_radius, 100)
z = np.zeros_like(r)

Ex, Ey, Ez = rw.compute_field(r, z)

# Normalize to total intensity
total_intensity = np.abs(Ex)**2 + np.abs(Ey)**2 + np.abs(Ez)**2
Ex_contribution = np.abs(Ex)**2 / total_intensity * 100
Ez_contribution = np.abs(Ez)**2 / total_intensity * 100

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

ax.plot(r, Ex_contribution, linewidth=3, label='|Ex|² contribution', color='tab:blue')
ax.plot(r, Ez_contribution, linewidth=3, label='|Ez|² contribution', color='tab:red')
ax.axhline(5, color='gray', linestyle='--', alpha=0.5, label='5% threshold')

ax.set_xlabel('Radial distance r (μm)')
ax.set_ylabel('Contribution to Total Intensity (%)')
ax.set_title(f'Field Component Contributions (NA={NA_high}, x-polarized)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 105])

plt.tight_layout()
plt.savefig(f'{output_dir}/field_components_high_NA.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nAt center (r=0):")
print(f"  Ex contribution: {Ex_contribution[0]:.1f}%")
print(f"  Ez contribution: {Ez_contribution[0]:.1f}%")
print(f"\nEz component is {Ez_contribution[0]:.1f}% of total - cannot be ignored!")